# CA1 — Building a Categorical Naive Bayes Classifier

The target columns are `poisonous` for Mushroom Dataset and `class` for Soybean Dataset.

## Part 3

In [25]:
import numpy as np
import pandas as pd

# TODO : fill each dataframe from csv files using pd.read_csv

full_test = pd.read_csv("mushroom_full_test.csv")
full_train = pd.read_csv("mushroom_full_train.csv")
small_train = pd.read_csv("mushroom_small_train.csv")
small_test = pd.read_csv("mushroom_small_test.csv")
soy_train = pd.read_csv("soybean_train.csv")
soy_test = pd.read_csv("soybean_test.csv")


In [26]:
TARGET_MUSHROOM = "poisonous"
TARGET_CLASSES = ["p", "f"]
SMALL_FEATURES = ["bruises", "gill-size"]

def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = len(subset_class) / total_samples

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = match_count / len(subset_class)
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]
    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)
                total_prob *= prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

small_model_direct = model_creation(small_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=SMALL_FEATURES)
small_predictions_direct = predict(
    small_model_direct, small_test[SMALL_FEATURES]
)
print(f"Small Mushroom baseline accuracy: "
      f"{accuracy(small_test[TARGET_MUSHROOM], small_predictions_direct):.2%}")

ALL_COLUMNS = [c for c in full_test.columns if c != TARGET_MUSHROOM]
big_model_direct = model_creation(full_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, full_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(full_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


Small Mushroom baseline accuracy: 76.80%
Full Mushroom baseline accuracy: 99.45%


مورد د : 

1: خیر دقت مدل در هر دو مجموعه داده های small full یکی نیست  

2: استفاده از ویژگی های بیشتر در داده ها و فرآیند یادگیری باعث افزایش میران دقت در تخمین پیش بینی را دارد  

3: خیر نمی توان این نتیجه را گرفت به این علت مه افزایش بیش از حد ویژگی باعص می شود که مدل این الگو ها رفتاری را یاد نگیرد بلکه داده ها و موقعیت ها را حفظ کند به همین سبب داده ها و ویژگی های کلید باید اضافه شوند و سایر ویژگی ها در نظر گرفته نشوند.

## Part 4

In [27]:
TARGET_MUSHROOM = "poisonous"
TARGET_CLASSES = ["p", "f"]
SMALL_FEATURES = ["bruises", "gill-size"]

def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = np.log(len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                # This part is for the exception of 0 in the np.log
                if match_count == 0:
                    prob = 0
                    print("Under flow has happened!!")
                else:
                    prob = np.log(match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)
                total_prob += prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

small_model_direct = model_creation(small_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=SMALL_FEATURES)
small_predictions_direct = predict(
    small_model_direct, small_test[SMALL_FEATURES]
)
print(f"Small Mushroom baseline accuracy: "
      f"{accuracy(small_test[TARGET_MUSHROOM], small_predictions_direct):.2%}")

ALL_COLUMNS = [c for c in full_test.columns if c != TARGET_MUSHROOM]
big_model_direct = model_creation(full_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, full_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(full_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


Small Mushroom baseline accuracy: 76.80%
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened

1: خبر دقت نسخه پیشین و لگاریتمی یکی نیست  
2: با توحه به خروحی داده شده وقتی این اختمال 0 می شود خروحی Underflow has happened خررا خواهیم داشت از طرفی  وقتی این پدیده را داریم  
برخی از حالت اختمال هایی که بهشان نسبت داده می شوند 0 می شود در این صورت مدل دقت واقغ بینانه ای نسب به پیش بینی نخواهد داشت.

## Part 5

In [28]:
soy_train.columns

Index(['date', 'plant-stand', 'precip', 'temp', 'hail', 'crop-hist',
       'area-damaged', 'severity', 'seed-tmt', 'germination', 'plant-growth',
       'leaves', 'leafspots-halo', 'leafspots-marg', 'leafspot-size',
       'leaf-shread', 'leaf-malf', 'leaf-mild', 'stem', 'lodging',
       'stem-cankers', 'canker-lesion', 'fruiting-bodies', 'external-decay',
       'mycelium', 'int-discolor', 'sclerotia', 'fruit-pods', 'fruit-spots',
       'seed', 'mold-growth', 'seed-discolor', 'seed-size', 'shriveling',
       'roots', 'class'],
      dtype='str')

فرآیند آموزش بدون لگاریتم

In [29]:
TARGET_MUSHROOM = "class"
ALL_COLUMNS = [c for c in soy_train.columns if c != TARGET_MUSHROOM]


def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = (len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = (match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model


def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    Total_zero_prob = 0
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)

                if prob == 0:
                    Total_zero_prob += 1

                    if Total_zero_prob < 3:
                        print(f"for the class {class_} the featur {feature} with the value {value} caused the o probability.")

                total_prob *= prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        if scores[best_class] == 0:
            print(f"index {_} : scores: {scores}")

        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation(soy_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, soy_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(soy_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


for the class 2-4-d-injury the featur date with the value 4.0 caused the o probability.
for the class 2-4-d-injury the featur plant-stand with the value 0.0 caused the o probability.
index 0 : scores: {'2-4-d-injury': np.float64(0.0), 'alternarialeaf-spot': np.float64(0.0), 'anthracnose': np.float64(0.0), 'bacterial-blight': np.float64(0.0), 'bacterial-pustule': np.float64(0.0), 'brown-spot': np.float64(0.0), 'brown-stem-rot': np.float64(0.0), 'charcoal-rot': np.float64(0.0), 'cyst-nematode': np.float64(0.0), 'diaporthe-pod-&-stem-blight': np.float64(0.0), 'diaporthe-stem-canker': np.float64(0.0), 'downy-mildew': np.float64(0.0), 'frog-eye-leaf-spot': np.float64(0.0), 'herbicide-injury': np.float64(0.0), 'phyllosticta-leaf-spot': np.float64(0.0), 'phytophthora-rot': np.float64(0.0), 'powdery-mildew': np.float64(0.0), 'purple-seed-stain': np.float64(0.0), 'rhizoctonia-root-rot': np.float64(0.0)}
index 1 : scores: {'2-4-d-injury': np.float64(0.0), 'alternarialeaf-spot': np.float64(0.0), 

آموزش داده با لگاریتم

In [30]:
TARGET_MUSHROOM = "class"
ALL_COLUMNS = [c for c in soy_train.columns if c != TARGET_MUSHROOM]


def model_creation_logarithm(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = np.log(len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = np.log(match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model


def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    Total_zero_probs = 0
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 1)
                if prob == 0:
                    Total_zero_probs += 1
                    prob = -(np.inf)

                    if Total_zero_probs < 3:
                        print(f"index {_} the feature {feature} with the value {value} caused the o probability.")
            
                total_prob += prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)

        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation_logarithm(soy_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, soy_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(soy_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3784831354.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3784831354.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3784831354.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3784831354.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3784831354.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3784831354.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count /

index 0 the feature leaves with the value 1 caused the o probability.
index 0 the feature hail with the value 0.0 caused the o probability.
Full Mushroom baseline accuracy: 0.00%


1: از دلایل ممکن برای این واقعه این هست که ممن کهست تعداد کلاس ها به حدی زیاد باشهکه این احتمالی که ممکن هست به هرکدوم از این کلاس ها برسه نزدیک به ضفر دربیاد   
مورد دیگری که هست این امکان وحود داره که به علت ویژگی های زیادی که یک نمونه داده می تونه داشته باشه با ضرب شدن این داده ها  
در هم دیگه باعث کم شدن بیش از حد این آمار بشه   
یک دلیل دیگر هم که ممکن هست باشد این هست که ویژگی در داده های test دیده بشود که قبلا در داده های train نبود و همین باعث شده اختمالی که  
به آن نسبت داده می شود 0 باشه و در صورت ضرب شدن با سایر احتمال ها کل این فرآیند احتمال را 0 بکند  
2: در مشکل Underflow ویژگی های دیده شده در داده های train test مشترک هستند و این 0 شدن احتمال بخاطر توالی موحود در بین ضرب شدن های   
احتمالی هست ولی این مشکل که در قسمت 1 رخ داد به علت نبود برخی از ویژگی هایی هست که در test حضور داشت ولی در train نبود.  
3: وقتی همه این داده های 0 هستند model کلاس به بر اساس خروف الفلا انتخاب می کند به این صورت که a مقدم تر از b می شود(البته این نکته هم هست که   
این روند در دیکشنری هست که داده های در اون ذخیره شدن و روند انتخاب در صورت برابری داده ها به همین صورت هست)

## Part6

In [31]:
TARGET_MUSHROOM = "class"
ALL_COLUMNS = [c for c in soy_train.columns if c != TARGET_MUSHROOM]

def get_unique_value(data_frame: pd.DataFrame):
    value_dictionary: dict = {}
    for column in ALL_COLUMNS:
        feature_value_list = data_frame[column].unique()
        value_dictionary[column] = feature_value_list
    return value_dictionary

value_dictionary: dict = get_unique_value(soy_train)

def model_creation_regularization(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list, value_dictionary: dict, alpha: float):
    model = {
        "priors": {},
        "conditionals": {},
        "class_counts": {},
        "alpha": alpha,
        "value_dictionary": value_dictionary
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        class_count = len(subset_class) 
        model["class_counts"][class_] = class_count
        model["priors"][class_] = class_count / total_samples

        for feature in SMALL_FEATURES:
            k_i = len(value_dictionary[feature])
            denominator = class_count + (alpha * k_i)
            
            for value in value_dictionary[feature]:
                match_count = (subset_class[feature] == value).sum()
                prob = (match_count + alpha) / denominator
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]
    alpha = model["alpha"]
    val_dict = model["value_dictionary"]

    predictions = []

    zero_probs_indexes: list = []
    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            class_count = model["class_counts"][class_]

            total_log_prob = (model["priors"][class_])

            for feature in features:
                value = row[feature]
                k_i = len(val_dict[feature])

                if (class_, feature, value) in model["conditionals"]:
                    prob = model["conditionals"][(class_, feature, value)]
                else:
                    prob = alpha / (class_count + alpha * (k_i + 1))

                total_log_prob *= (prob)

            scores[class_] = total_log_prob

        best_class = max(scores, key=scores.get)

        # Printing the probs wich are 0
        if scores[best_class] == 0 :
            zero_probs_indexes.append(_)

        predictions.append(best_class)
    print(zero_probs_indexes)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)
alpha_list: list = [0, 0.01, 0.05, 0.1]
for a in alpha_list:
    print("-----------------------------------------------------------")

    big_model_direct = model_creation_regularization(
        small_train=soy_train, 
        TARGET_COLUMN=TARGET_MUSHROOM, 
        SMALL_FEATURES=ALL_COLUMNS, 
        value_dictionary=value_dictionary, 
        alpha=a
    )

    full_predictions_direct = predict(
        model=big_model_direct, 
        sample_frame=soy_test[ALL_COLUMNS]
    )

    print(f"Full Mushroom baseline accuracy: "
        f"{accuracy(soy_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")

    print("-----------------------------------------------------------")


-----------------------------------------------------------
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72]
Full Mushroom baseline accuracy: 0.00%
-----------------------------------------------------------
-----------------------------------------------------------
[]
Full Mushroom baseline accuracy: 86.30%
-----------------------------------------------------------
-----------------------------------------------------------
[]
Full Mushroom baseline accuracy: 78.08%
-----------------------------------------------------------
-----------------------------------------------------------
[]
Full Mushroom baseline accuracy: 76.71%
-----------------------------------------------------------


1: مقدار 0.01 برای آلفا دقت بهتری نسبت به سایر مقادیر آلفا یاحاد کرده  
2: با افزایش مقادیر آلفا تعداد مقادیری که دارای احتمال 0 هستند کاهش پیدا می کند  
3: خیر این گزاره دست نیست در صورت اینکه این مقدار آلفا به سمت بی نهیت میل کند در این صورت این مقدار به یک تایع با احتمال برابر میان  
کلاس ها میل می کند درواقع این کلاس ها خاصیت تمایز را از دست می دهند.از لخاط ریاضی هم به قضیه نگاه کنیم اگه این آلفا به سمت بی نهای تمیل  
و بیایم صورت و مخرحش رو به آلفا تقسیم کنیم می بینیم که عملا داره به سمت تابغ یونیفرم میره  
4: چون برای هر داده و دسته این مقداری از آلفا که می تواند بیشترین مقدار دقت را تولید کند مختلف هست و برای همین نیاز هیت که به عنوان 
یه پارامتر ثابل یادگیری ازش استفاده بشه

## Part 7

In [32]:
results_of_test: dict = {}

Normal without regularization

In [33]:
TARGET_COLUMN = "poisonous"
ALL_COLUMNS = [c for c in full_train.columns if c != TARGET_COLUMN]


def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = (len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = (match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_COLUMN]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)
                total_prob *= prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        if scores[best_class] == 0:
            print(f"index {_} : scores: {scores}")

        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation(full_train, TARGET_COLUMN=TARGET_COLUMN, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, full_test[ALL_COLUMNS]
)

accuracy_score = accuracy(full_test[TARGET_COLUMN], full_predictions_direct)
results_of_test["normal_without_regularization"] = accuracy_score

Logarithm without regularization

In [34]:
def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = (len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = np.log(match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_COLUMN]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = np.log(model["conditionals"].get((class_, feature, value), 0.0))
                total_prob += prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        if scores[best_class] == 0:
            print(f"index {_} : scores: {scores}")

        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation(full_train, TARGET_COLUMN=TARGET_COLUMN, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, full_test[ALL_COLUMNS]
)

accuracy_score = accuracy(full_test[TARGET_COLUMN], full_predictions_direct)
results_of_test["logarithm_without_regularization"] = accuracy_score

C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3400215290.py:18: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3400215290.py:18: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3400215290.py:18: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3400215290.py:18: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3400215290.py:18: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_20460\3400215290.py:18: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count /

Normal with regularization

In [35]:
def get_unique_value(data_frame: pd.DataFrame):
    value_dictionary: dict = {}
    for column in ALL_COLUMNS:
        feature_value_list = data_frame[column].unique()
        value_dictionary[column] = feature_value_list
    return value_dictionary

value_dictionary: dict = get_unique_value(full_train)

def model_creation_regularization(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list, value_dictionary: dict, alpha: float):
    model = {
        "priors": {},
        "conditionals": {},
        "class_counts": {},
        "alpha": alpha,
        "value_dictionary": value_dictionary
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        class_count = len(subset_class) 
        model["class_counts"][class_] = class_count
        model["priors"][class_] = class_count / total_samples

        for feature in SMALL_FEATURES:
            k_i = len(value_dictionary[feature])
            denominator = class_count + (alpha * k_i)
            
            for value in value_dictionary[feature]:
                match_count = (subset_class[feature] == value).sum()
                prob = (match_count + alpha) / denominator
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_COLUMN]
    alpha = model["alpha"]
    val_dict = model["value_dictionary"]

    predictions = []

    zero_probs_indexes: list = []
    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            class_count = model["class_counts"][class_]

            total_log_prob = (model["priors"][class_])

            for feature in features:
                value = row[feature]
                k_i = len(val_dict[feature])

                if (class_, feature, value) in model["conditionals"]:
                    prob = model["conditionals"][(class_, feature, value)]
                else:
                    prob = alpha / (class_count + alpha * (k_i + 1))

                total_log_prob *= (prob)

            scores[class_] = total_log_prob

        best_class = max(scores, key=scores.get)

        # Printing the probs wich are 0
        if scores[best_class] == 0 :
            zero_probs_indexes.append(_)

        predictions.append(best_class)
    print(zero_probs_indexes)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation_regularization(
    small_train=full_train, 
    TARGET_COLUMN=TARGET_COLUMN, 
    SMALL_FEATURES=ALL_COLUMNS, 
    value_dictionary=value_dictionary, 
    alpha=0.01
)

full_predictions_direct = predict(
    model=big_model_direct, 
    sample_frame=full_test[ALL_COLUMNS]
)

accuracy_score = accuracy(full_test[TARGET_COLUMN], full_predictions_direct)
results_of_test["normal_with_regularization"] = accuracy_score


[]


Logarithm with regularization

In [36]:
def get_unique_value(data_frame: pd.DataFrame):
    value_dictionary: dict = {}
    for column in ALL_COLUMNS:
        feature_value_list = data_frame[column].unique()
        value_dictionary[column] = feature_value_list
    return value_dictionary

value_dictionary: dict = get_unique_value(full_train)

def model_creation_regularization(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list, value_dictionary: dict, alpha: float):
    model = {
        "priors": {},
        "conditionals": {},
        "class_counts": {},
        "alpha": alpha,
        "value_dictionary": value_dictionary
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        class_count = len(subset_class) 
        model["class_counts"][class_] = class_count
        model["priors"][class_] = class_count / total_samples

        for feature in SMALL_FEATURES:
            k_i = len(value_dictionary[feature])
            denominator = class_count + (alpha * k_i)
            
            for value in value_dictionary[feature]:
                match_count = (subset_class[feature] == value).sum()
                prob = (match_count + alpha) / denominator
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_COLUMN]
    alpha = model["alpha"]
    val_dict = model["value_dictionary"]

    predictions = []

    zero_probs_indexes: list = []
    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            class_count = model["class_counts"][class_]

            total_log_prob = (model["priors"][class_])

            for feature in features:
                value = row[feature]
                k_i = len(val_dict[feature])

                if (class_, feature, value) in model["conditionals"]:
                    prob = np.log(model["conditionals"][(class_, feature, value)])
                else:
                    prob = np.log(alpha / (class_count + alpha * (k_i + 1)))

                total_log_prob += (prob)

            scores[class_] = total_log_prob

        best_class = max(scores, key=scores.get)

        # Printing the probs wich are 0
        if scores[best_class] == 0 :
            zero_probs_indexes.append(_)

        predictions.append(best_class)
    print(zero_probs_indexes)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation_regularization(
    small_train=full_train, 
    TARGET_COLUMN=TARGET_COLUMN, 
    SMALL_FEATURES=ALL_COLUMNS, 
    value_dictionary=value_dictionary, 
    alpha=0.01
)

full_predictions_direct = predict(
    model=big_model_direct, 
    sample_frame=full_test[ALL_COLUMNS]
)

accuracy_score = accuracy(full_test[TARGET_COLUMN], full_predictions_direct)
results_of_test["loagarithm_with_regularization"] = accuracy_score


[]


In [37]:
print("For the regularization the alpha is  : 0.01 ->")
for obj in results_of_test:
    print(f"{obj} with score : {results_of_test[obj]}")

For the regularization the alpha is  : 0.01 ->
normal_without_regularization with score : 0.9944615384615385
logarithm_without_regularization with score : 0.48184615384615387
normal_with_regularization with score : 0.9895384615384616
loagarithm_with_regularization with score : 0.9895384615384616


1: بله دقت مدل با اضافه کردن این هموار سازی تعیر کرده است   
2, 3: احتمال می دم بر اساس نوع توزیعی باشه داده اتخاذ می کنه مثلا اگه داده های ما دارای داده های پرت خیلی زیادی باشن  
و یا از نوع توزیعی مه این روش  به صورت پیش فرض ازش استفاده کرده رو نداشته باشن در این صورت این  روش ممکنه براش باغث ایحاد خطا بشه    
4: در روش لگاریتمی ما احتمال هامون 0 نست بلکه خیلی کم و نزذیک به 0 هست که در صورت ضرب شدن این عبارت نهایی رو کم می کنهولی در روش هموار   
سازی با این احتمالمون 0 هست و باغث میشه اثرات این احتمال های قبلی هم از بین بره.